# Fall 2024 Data Science Track: Week 2 - Data Cleaning Exercise

## Packages, Packages, Packages!

Import *all* the things here! You need the standard stuff: `pandas` and `numpy`.

If you got more stuff you want to use, add them here too. 🙂

In [1]:
# Install pandas and numpy if the active Python environment does not have them. Using Python venv is recommended when you are doing so.
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import here.
import pandas as pd
import numpy as np

pd.set_option('display.width', 200)

## Introduction

With the packages out of the way, now you will be working with the following data sets:

* `food_coded.csv`: [Food choices](https://www.kaggle.com/datasets/borapajo/food-choices?select=food_coded.csv) from Kaggle
* `Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv`: [Ask A Manager Salary Survey 2021 (Responses)](https://docs.google.com/spreadsheets/d/1IPS5dBSGtwYVbjsfbaMCYIWnOuRmJcbequohNxCyGVw/view?&gid=1625408792) as *Tab Separated Values (.tsv)* from Google Docs

Each one poses different challenges. But you’ll―of course―overcome them with what you learned in class! 😉

## Food Choices Data Set

### Load the Data

Load the Food choices data set into a new variable, `df_food`.

In [3]:
# Load the Food choices data set.

food_data_set_path = '../data/food_coded.csv'

df_food = pd.read_csv(food_data_set_path)

### Explore the Data

How much data did you just load?

In [4]:
# Count by hand. (lol kidding)
df_food.shape

(125, 61)

In [5]:
# Try to print or display it nicely.
pd.DataFrame([df_food.shape], columns=['rows', 'columns'], index=['df_food'])

,rows,columns
df_food,125,61


What are the columns and their types in this data set?

In [6]:
# Show the column names and their types.
df_food.dtypes

GPA                     str
Gender                int64
breakfast             int64
calories_chicken      int64
calories_day        float64
                     ...   
type_sports             str
veggies_day           int64
vitamins              int64
waffle_calories       int64
weight                  str
Length: 61, dtype: object

In [7]:
pd.set_option("display.max_rows", 100) # Set pandas to render up to 100 rows first. Default caps to 10 and takes the middle chunk out to make the DataFrame fit.

# Try to display the column information as a nice pandas DataFrame.
df_food.dtypes.to_frame(name='dtype').rename_axis('column')

,dtype
column,
GPA,str
Gender,int64
breakfast,int64
calories_chicken,int64
calories_day,float64
calories_scone,float64
coffee,int64
comfort_food,str
comfort_food_reasons,str


### Clean the Data

Perhaps we’d like to know more another day, but the team is really interested in just the relationship between calories (`calories_day`) and weight. …and maybe gender.

Can you remove the other columns? (Assign the result to a new variable, `df_food_col_subset`.)

In [8]:
# Remove ‘em.
columns_we_want = ['calories_day', 'weight', 'Gender']

df_food_col_subset = df_food[columns_we_want]
df_food_col_subset.head()

,calories_day,weight,Gender
0,NaN,187,2
1,3.0,155,1
2,4.0,I'm not answering this.,1
3,3.0,"Not sure, 240",1
4,2.0,190,1


In [9]:
# Is there a second way to remove columns?
columns_we_do_not_want = df_food.columns.difference(columns_we_want)

df_food.drop(columns=columns_we_do_not_want).head()

,Gender,calories_day,weight
0,2,NaN,187
1,1,3.0,155
2,1,4.0,I'm not answering this.
3,1,3.0,"Not sure, 240"
4,1,2.0,190


In [10]:
# 🚀 Extra credit: What about a third way to remove columns?
df_food.filter(items=columns_we_want).head()

,calories_day,weight,Gender
0,NaN,187,2
1,3.0,155,1
2,4.0,I'm not answering this.,1
3,3.0,"Not sure, 240",1
4,2.0,190,1


What about `NaN`s? How many are there?

In [11]:
# Count ‘em.
df_food_col_subset.isna().sum()

calories_day    19
weight           2
Gender           0
dtype: int64

In [12]:
# 🚀 Extra credit: Try to display the NaN sums as a nice pandas DataFrame.
df_food_col_subset.isna().sum().to_frame(name='nan_count').rename_axis('column')

,nan_count
column,
calories_day,19
weight,2
Gender,0


In [13]:
# 🚀 Extra credit: You can turn one line of code into multiple lines of code using a backslash as a continuation character to break up what would otherwise be a long line.
nan_summary = df_food_col_subset.isna() \
    .sum() \
    .to_frame(name='nan_count') \
    .rename_axis('column') \
    .sort_values('nan_count', ascending=False)

nan_summary

,nan_count
column,
calories_day,19
weight,2
Gender,0


We gotta remove those `NaN`s―the entire row.

In [14]:
# Drop ‘em in-place.
df_food_col_subset.dropna(inplace=True)

print(f'{df_food_col_subset.shape[0]} rows left')
df_food_col_subset.isna().sum()

104 rows left


calories_day    0
weight          0
Gender          0
dtype: int64

In [15]:
# 🚀 Extra credit: Check if the in-place modifications also affected the original DataFrame, df_food. 🙂
print(f'df_food_col_subset: {df_food_col_subset.shape[0]} rows')
print(f'df_food:            {df_food.shape[0]} rows')
print(f'df_food NaNs in calories_day: {df_food["calories_day"].isna().sum()}')
print()
print('df_food was NOT affected: selecting columns returned a copy, not a view,')
print('so dropna(inplace=True) only touched df_food_col_subset.')

df_food_col_subset: 104 rows
df_food:            125 rows
df_food NaNs in calories_day: 19

df_food was NOT affected: selecting columns returned a copy, not a view,
so dropna(inplace=True) only touched df_food_col_subset.


But what about the weird non-numeric values in the column obviously meant for numeric data?

Notice the data type of that column from when you got the types of all the columns?

If only we could convert the column to a numeric type and drop the rows with invalid values. 🤔

In [16]:
# Fix that in-place.
print('junk values:', [w for w in df_food_col_subset['weight'].unique() if not str(w).isdigit()])

df_food_col_subset['weight'] = pd.to_numeric(df_food_col_subset['weight'], errors='coerce')
df_food_col_subset.dropna(subset=['weight'], inplace=True)

print(f'{df_food_col_subset.shape[0]} rows left')
df_food_col_subset.dtypes

junk values: ["I'm not answering this. ", 'Not sure, 240', '144 lbs']
101 rows left


calories_day    float64
weight          float64
Gender            int64
dtype: object

Now this data seems reasonably clean for our purposes! 😁

Let’s save it somewhere to be shipped off to another teammate. 💾

In [17]:
# Savey save!
clean_food_path = '../data/food_choices_clean.csv'

df_food_col_subset.to_csv(clean_food_path, index=False)
pd.read_csv(clean_food_path).head()

,calories_day,weight,Gender
0,3.0,155.0,1
1,2.0,190.0,1
2,3.0,190.0,1
3,3.0,180.0,2
4,3.0,137.0,1


In [18]:
# 🚀 Extra credit: Load and print a few lines of the saved file as plain text.
with open(clean_food_path) as file:
    for line_number, line in enumerate(file):
        if line_number >= 5:
            break
        print(line.rstrip())

calories_day,weight,Gender
3.0,155.0,1
2.0,190.0,1
3.0,190.0,1
3.0,180.0,2


## Ask a Manager Salary Survey 2021 (Responses) Data Set

### Load the Data

Load the Ask A Manager Salary Survey 2021 (Responses) data set into a new variable, `df_salary`.

In [19]:
# Load the Ask A Manager Salary Survey 2021 (Responses) data set.
salary_data_set_path = '../data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv'

df_salary = pd.read_csv(salary_data_set_path, sep='\t')
df_salary.head()

,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,4/27/2021 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,"55,000",0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,4/27/2021 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,"34,000",NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White
3,4/27/2021 11:02:41,25-34,Nonprofits,Program Manager,NaN,"62,000",3000.0,USD,NaN,NaN,USA,Wisconsin,Milwaukee,8 - 10 years,5-7 years,College degree,Woman,White
4,4/27/2021 11:02:42,25-34,"Accounting, Banking & Finance",Accounting Manager,NaN,"60,000",7000.0,USD,NaN,NaN,US,South Carolina,Greenville,8 - 10 years,5-7 years,College degree,Woman,White


Was that hard? 🙃

### Explore

You know the drill.

How much data did you just load?

In [20]:
# Count by hand. I’m dead serious.
pd.DataFrame([df_salary.shape], columns=['rows', 'columns'], index=['df_salary'])

,rows,columns
df_salary,28062,18


What are the columns and their types?

In [21]:
# Show the column names and their types.
df_salary.dtypes.to_frame(name='dtype').rename_axis('column')

,dtype
column,
Timestamp,str
How old are you?,str
What industry do you work in?,str
Job title,str
"If your job title needs additional context, please clarify here:",str
"What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)",str
"How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",float64
Please indicate the currency,str
"If ""Other,"" please indicate the currency here:",str


Oh… Ugh! Give these columns easier names to work with first. 🙄

In [22]:
# Rename ‘em in-place.
# Non-binding suggestions: timestamp, age, industry, title, title_context, salary, additional_compensation, currency, other_currency, salary_context, country, state, city, total_yoe, field_yoe, highest_education_completed	gender, race
new_column_names = [
    'timestamp',
    'age',
    'industry',
    'title',
    'title_context',
    'salary',
    'additional_compensation',
    'currency',
    'other_currency',
    'salary_context',
    'country',
    'state',
    'city',
    'total_yoe',
    'field_yoe',
    'highest_education_completed',
    'gender',
    'race',
]

df_salary.rename(columns=dict(zip(df_salary.columns, new_column_names)), inplace=True)
df_salary.dtypes

timestamp                          str
age                                str
industry                           str
title                              str
title_context                      str
salary                             str
additional_compensation        float64
currency                           str
other_currency                     str
salary_context                     str
country                            str
state                              str
city                               str
total_yoe                          str
field_yoe                          str
highest_education_completed        str
gender                             str
race                               str
dtype: object

It’s a lot, and that should not have been easy. 😏

You’re going to have a gander at the computing/tech subset first because thats *your* industry. But first, what value corresponds to that `industry`?

In [23]:
# List the unique industries and a count of their instances.
df_salary['industry'].value_counts().head(10)

industry
Computing or Tech                       4699
Education (Higher Education)            2464
Nonprofits                              2419
Health care                             1896
Government and Public Administration    1889
Accounting, Banking & Finance           1809
Engineering or Manufacturing            1695
Marketing, Advertising & PR             1133
Law                                     1097
Business or Consulting                   852
Name: count, dtype: int64

That value among the top 5 is what you’re looking for innit? Filter out all the rows not in that industry and save it into a new variable, `df_salary_tech`. 

In [24]:
# Filtery filter.
df_salary_tech = df_salary[df_salary['industry'] == 'Computing or Tech'].copy()

df_salary_tech.head()

,timestamp,age,industry,title,title_context,salary,additional_compensation,currency,other_currency,salary_context,country,state,city,total_yoe,field_yoe,highest_education_completed,gender,race
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
8,4/27/2021 11:03:01,45-54,Computing or Tech,Systems Analyst,Data developer/ETL Developer,"112,000",10000.0,USD,NaN,NaN,US,Missouri,St. Louis,21 - 30 years,21 - 30 years,College degree,Woman,White
43,4/27/2021 11:04:04,25-34,Computing or Tech,Principal Software Engineer,NaN,"187,500",5000.0,USD,NaN,NaN,United States,Pennsylvania,Pittsburgh,8 - 10 years,5-7 years,College degree,Woman,White
44,4/27/2021 11:04:04,25-34,Computing or Tech,Intelligence Analyst,NaN,"110,000",20000.0,USD,NaN,"Around 20,000 a year in stock",USA,Virginia,"Arlington, VA",8 - 10 years,8 - 10 years,Master's degree,Man,White
46,4/27/2021 11:04:07,35-44,Computing or Tech,Mobile developer,NaN,"144,600",2500.0,USD,NaN,NaN,USA,Massachusetts,Boston,5-7 years,5-7 years,PhD,Woman,White


Do a sanity check by counting.

In [25]:
# Sanity check count.
df_salary_tech['industry'].value_counts()

industry
Computing or Tech    4699
Name: count, dtype: int64

We are very interested in salary figures. But how many dollars 💵 is a euro 💶 or a pound 💷? That sounds like a problem for another day. 🫠

For now, let’s just look at U.S. dollars (`'USD'`).

In [26]:
# Filtery filter (in place) for just the jobs that pay in USD!
df_salary_tech.drop(
    index=df_salary_tech[df_salary_tech['currency'] != 'USD'].index,
    inplace=True,
)

print(f'{df_salary_tech.shape[0]} rows left')
df_salary_tech['currency'].value_counts()

3777 rows left


currency
USD    3777
Name: count, dtype: int64

What we really want know is how each U.S. state pays in tech. What value in `country` represents the United States of America?

In [27]:
# We did filter for USD, so if we do a count of each unique country in descending count order, the relevant value(s) should show up at the top.
df_salary_tech['country'].value_counts().head(10)

country
United States               1576
USA                         1222
US                           412
U.S.                         108
United States of America      90
United States                 68
Usa                           59
USA                           56
usa                           28
United states                 23
Name: count, dtype: int64

### Clean the Data

Well, we can’t get our answers with what we currently have, so you’ll have to make some changes.

Let’s not worry about anything below the first 5 values for now. Convert the top 5 to a single canonical value―say, `'US'`, which is nice and short.

In [28]:
# Replace them all in-place with 'US'.
top_5_countries = df_salary_tech['country'].value_counts().head(5).index.tolist()
print('replacing:', top_5_countries)

df_salary_tech.replace({'country': {name: 'US' for name in top_5_countries}}, inplace=True)
df_salary_tech['country'].head()

replacing: ['United States', 'USA', 'US', 'U.S.', 'United States of America']


8     US
43    US
44    US
46    US
47    US
Name: country, dtype: str

Have a look at the count of each unique country again now.

In [29]:
# Count again.
df_salary_tech['country'].value_counts().head(20)

country
US                           3408
United States                  68
Usa                            59
USA                            56
usa                            28
United states                  23
united states                  14
Us                             12
us                              9
U.S.A.                          7
United States of America        7
Israel                          5
Canada                          4
U.S.                            2
United State of America         2
Unite States                    2
Australia                       2
UnitedStates                    2
India                           2
U.S                             2
Name: count, dtype: int64

Did you notice anything interesting?

In [30]:
# 🚀 Extra credit: Resolve [most of] those anomalous cases too without exhaustively taking every variant literally into account.
normalized_country = (
    df_salary_tech['country']
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z]', '', regex=True)
)

is_united_states = normalized_country.str.fullmatch(r'us|usa|america|uni\w*sta\w*').fillna(False)

df_salary_tech['country'] = df_salary_tech['country'].where(~is_united_states, 'US')
print(f'{is_united_states.sum()} of {len(df_salary_tech)} rows resolved to US')

3723 of 3777 rows resolved to US


In [31]:

# 🚀 Extra credit: If you’ve resolved it, let’s see how well you did by counting the number of instances of each unique value.
df_salary_tech['country'].value_counts().head(20)

country
US                      3723
Israel                     5
Canada                     4
Australia                  2
India                      2
Spain                      2
Brazil                     2
United Kingdom             2
New Zealand                2
Poland                     2
France                     2
Puerto Rico                1
Cuba                       1
Danmark                    1
Italy                      1
International              1
Remote (philippines)       1
Singapore                  1
Uruguay                    1
Mexico                     1
Name: count, dtype: int64

It’s looking good so far. Let’s find out the minimum, mean, and maximum (in that order) salary by state, sorted by the mean in descending order.

In [32]:
# Find the minimum, mean, and maximum salary in USD by U.S. state.
try:
    df_salary_tech.groupby('state')['salary'].agg(['min', 'mean', 'max']).sort_values('mean', ascending=False)
except TypeError as error:
    print(f'{type(error).__name__}: {error}')

TypeError: dtype 'str' does not support operation 'mean'


 Well, pooh! We forgot that `salary` isn’t numeric. Something wrong must be fixed. 🤔

In [33]:
# Fix it in-place.
print('before:', df_salary_tech['salary'].head(3).tolist(), '->', df_salary_tech['salary'].dtype)

df_salary_tech['salary'] = pd.to_numeric(
    df_salary_tech['salary'].str.replace(',', '', regex=False),
    errors='coerce',
)
df_salary_tech.dropna(subset=['salary'], inplace=True)

print('after: ', df_salary_tech['salary'].head(3).tolist(), '->', df_salary_tech['salary'].dtype)

before: ['112,000', '187,500', '110,000'] -> str
after:  [112000, 187500, 110000] -> int64


Let’s try that again.

In [34]:
# Try it again. Yeah!
salary_by_state = (
    df_salary_tech[df_salary_tech['country'] == 'US']
    .groupby('state')['salary']
    .agg(['min', 'mean', 'max'])
    .sort_values('mean', ascending=False)
)
salary_by_state

,min,mean,max
state,,,
"Michigan, Texas, Washington",340000,340000.000000,340000
"California, Oregon",200000,200000.000000,200000
"California, Colorado",176000,176000.000000,176000
"Georgia, Massachusetts",175000,175000.000000,175000
Florida,28800,157457.232143,2600000
"Alabama, District of Columbia",156000,156000.000000,156000
California,0,155019.612462,875000
Washington,72,151423.670588,950000
New York,14000,148054.638968,590000


That did the trick! Now let’s narrow this to data 2021 and 2022 just because (lel). *(Hint: that timestamp column may not be a temporal type right now.)*

In [35]:
# Filter the data to within 2021, 2022, or 2023, saving the DataFrame to a new variable, and generate the summary again.
df_salary_tech['timestamp'] = pd.to_datetime(df_salary_tech['timestamp'], format='%m/%d/%Y %H:%M:%S')

df_salary_tech_recent = df_salary_tech[df_salary_tech['timestamp'].dt.year.isin([2021, 2022, 2023])]
print(f'{df_salary_tech_recent.shape[0]} of {df_salary_tech.shape[0]} rows kept')

salary_by_state_recent = (
    df_salary_tech_recent[df_salary_tech_recent['country'] == 'US']
    .groupby('state')['salary']
    .agg(['min', 'mean', 'max'])
    .sort_values('mean', ascending=False)
)
salary_by_state_recent

3764 of 3777 rows kept


,min,mean,max
state,,,
"Michigan, Texas, Washington",340000,340000.000000,340000
"California, Oregon",200000,200000.000000,200000
"California, Colorado",176000,176000.000000,176000
"Georgia, Massachusetts",175000,175000.000000,175000
"Alabama, District of Columbia",156000,156000.000000,156000
California,0,155149.018265,875000
Washington,72,151571.970501,950000
New York,14000,148054.638968,590000
Nevada,38000,141310.000000,425000


## Bonus

Clearly, we do not have enough data to produce useful figures for the level of specificity you’ve now reached. What do you notice about Delaware and West Virginia?

Let’s back out a bit and return to `df_salary` (which was the loaded data with renamed columns but *sans* filtering).

### Bonus #0

Apply the same steps as before to `df_salary`, but do not filter for any specific industry. Do perform the other data cleaning stuff, and get to a point where you can generate the minimum, mean, and maximum by state.

### Bonus #1

This time, format the table output nicely (*$12,345.00*) without modifying the values in the `DataFrame`. That is, `df_salary` should be identical before versus after running your code.

(*Hint: if you run into an error about `jinja2` perhaps you need to `pip install` something.*)

### Bonus #2

Filter out the non-single-states (e.g., `'California, Colorado'`) in the most elegant way possible (i.e., *not* by blacklisting all the bad values).

### Bonus #3

Show the quantiles instead of just minimum, mean, and maximum―say 0%, 5%, 25%, 50%, 75%, 95%, and 100%. Outliers may be deceiving.

Sort by whatever interests you―like say the *50th* percentile.

And throw in a count by state too. It would be interesting to know how many data points contribute to the figures for each state. (*Hint: your nice formatting from Bonus #1 might not work this time around.* 😜)